# Session 5: Introduction to MCP

**Learning goals:**

- Explain how MCP reduces repeated tool integration work in AI applications.
- Identify the host, client and server, and distinguish tools, resources and prompts.
- Compare stdio with Streamable HTTP and trace a tool discovery and invocation exchange.
- Build a weather MCP server and connect a Python client to it.

**Prerequisites:**

- Python functions, dictionaries, JSON, HTTP and Function Calling.
- Use a WeatherAPI key for the custom weather tool.
- Have an LLM API key and a model that supports Function Calling.


## Outline

1. From Function Calling to MCP
2. MCP architecture and core primitives
3. Transport in MCP
4. Build your own MCP server
5. Connect MCP to the chatbot
6. Multi-tool MCP chatbot


## 1. From Function Calling to MCP

In [Sessions 3–4](<Session 3-4 Building a Chatbot with an LLM API.ipynb>), we implemented a weather function, wrote its JSON Schema, and added a function mapping and dispatch code. Each application needed its own tool integration code. With more applications and tools, that work was repeated.

The **Model Context Protocol (MCP)** provides a common interface for discovering and invoking external capabilities. Tool implementations live in reusable servers, which compatible applications access through MCP clients.

- **Discover tools:** `tools/list` returns tool names, descriptions and input schemas. FastMCP can generate these schemas from Python type hints and docstrings.
- **Call tools:** `tools/call` sends a tool name and arguments to the server for execution.
- **Reuse implementations:** multiple applications can connect to the same tool server instead of maintaining separate copies of its service-specific code.

**Function Calling and MCP work together:** the model proposes a tool call; the host checks permissions and routes it through MCP. The host still converts tool descriptions to its LLM provider's format and returns results with the matching call ID.

In the figure, both designs use the same two APIs. Without MCP, two applications maintain four adapter implementations. With MCP, they reuse two adapters held in shared servers. The orange blocks show where the service-specific code lives.

[Official MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)


![Applications duplicate service-specific adapters without MCP; with MCP, they reuse adapters in shared servers.](assets/session5/before-after-mcp.png)


## 2. MCP architecture

| Role | Responsibility | Connection to earlier sessions |
|---|---|---|
| Host | Owns the user interface, LLM conversation, server selection and permissions | The chatbot application from Sessions 3–4 |
| Client | Handles protocol communication with one server | A protocol-aware connection inside the chatbot |
| Server | Publishes and implements tools, resources and prompts | Similar service responsibility to Session 2's FastAPI backend |

One host can create several clients, each with a one-to-one connection to a server. A server process can also serve other clients when its transport and deployment support that. The LLM does not directly execute Python: the host routes an allowed request through a client.

![An AI host contains two clients, each connected to one server exposing tools, resources and prompts.](assets/session5/mcp-architecture.png)

In the diagram, Client A connects to the weather server over stdio. Client B connects to an illustrative document server over Streamable HTTP, labelled HTTP in the figure. Each server groups its capabilities into tools, resources and prompts. A server may expose any supported subset; it does not have to provide all three. The weather examples match the server we build later; the document server illustrates a second connection. Arrows from the LLM represent requests routed by the host, which controls execution.

The host controls which servers and data enter a conversation. For example, a document server and a weather server need separate permissions even when they share the same protocol. A server can wrap a REST API internally; MCP does not replace that upstream API.

The practical weather example below uses HTTP so the server and client can run in separate terminals.


### 2.1 Tools, resources and prompts

| Primitive | What it offers | Common control pattern | Example |
|---|---|---|---|
| Tool | An executable operation, including reads or writes | Model proposes; host authorises and invokes | `get_weather(city="Perth")` |
| Resource | Addressable context read by URI | Application selects context | `info://server-status` |
| Prompt | A reusable message template with arguments | User selects a workflow through the host | A weather briefing template |

Resources provide data without a write operation; their content can still change over time. A resource URI identifies content and need not be a web URL. Resources may be fixed (`info://server-status`) or templated (`docs://documents/{doc_id}`). Text, JSON and binary data use appropriate MIME types.

A prompt can produce one or more messages. Retrieving it does not call the LLM or run its suggested tools. The host decides how to use those messages.

Tools are our main focus because they connect to the previous chatbot. We will also read one resource and retrieve one prompt to make the distinctions concrete.

[MCP server concepts](https://modelcontextprotocol.io/docs/learn/server-concepts)


![The MCP client invokes tools, reads resources and retrieves prompts exposed by a server; each capability is illustrated with examples.](assets/session5/mcp-client-server-primitives.png)


## 3. Transport in MCP

A **transport** handles how messages are sent and received between an MCP client and server.

1. **For servers running locally:**
   - **stdio (standard input/output):** the client starts the server as a subprocess and exchanges messages through its standard input and output.

2. **For remote servers:**
   - **HTTP+SSE (Server-Sent Events):** the older transport from protocol version **2024-11-05**, using HTTP POST for client messages and a separate SSE connection for server messages.
   - **Streamable HTTP:** introduced in protocol version **2025-03-26**, using one MCP endpoint with JSON or SSE responses. It replaces HTTP+SSE.

HTTP servers can also run locally, as in the two-terminal examples.

[Official MCP transport specification](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports)

![Compare local subprocess pipes and an independent HTTP server.](assets/session5/mcp-transport.png)

**Communication lifecycle: two protocol versions**

Both versions use `tools/list` to discover available tools and `tools/call` to invoke one. The main difference is how protocol information is exchanged before and during those requests.

- **MCP 2025-11-25:** the client first sends `initialize`, receives the server's version and capabilities, then sends `notifications/initialized`. It can then list and call tools. Communication ends by closing the transport.
- **MCP 2026-07-28:** there is no required initialisation handshake. The client can optionally call `server/discover`; each request carries its protocol version and client capabilities. HTTP requests complete independently, with no protocol session to terminate. A local stdio connection ends by closing its pipes.

Clients supporting both versions can use the older handshake when connecting to a legacy server. The lower row is a teaching sequence, rather than three mandatory protocol phases.

[Session-based flow (2025-11-25)](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle) · [Per-request metadata and compatibility (2026-07-28)](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning)

![MCP 2025-11-25 uses an initialisation handshake; MCP 2026-07-28 uses per-request metadata. Both versions list and call tools.](assets/session5/mcp-communication-lifecycle.png)


## 4. Build your own MCP server

Install the required libraries in your Python environment:

```bat
python -m pip install fastmcp requests
```

Use the existing WeatherAPI request in `get_weather(city)` to return `city`, `current_temp` and `condition`. The `@mcp.tool()` decorator exposes the function through MCP and generates its input schema.

The code below defines the server, including a status resource and briefing prompt. Set `WEATHER_API_KEY` in the Python file before calling the weather tool. Running this cell defines the capabilities; start the HTTP server in Terminal 1 using the instructions below.


In [ ]:
"""Expose WeatherAPI through a local MCP server for Session 5."""

import requests
from fastmcp import FastMCP

WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"  # Replace with your WeatherAPI key.
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"
mcp = FastMCP(name="Course Weather Server")


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return simplified weather for a city using WeatherAPI.

    Args:
        city: A city name, such as Perth or Sydney; include the country if needed.
    """
    city = city.strip()
    if not city:
        raise ValueError("Enter a city name.")
    api_key = WEATHER_API_KEY.strip()
    if not api_key or api_key == "YOUR_WEATHERAPI_KEY":
        raise ValueError("Replace WEATHER_API_KEY in weather_mcp_server.py before calling get_weather.")

    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": api_key, "q": city, "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL, params=params, timeout=10  # Wait up to 10 seconds.
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


@mcp.resource("info://server-status", mime_type="text/plain")
def server_status() -> str:
    """Describe this server without making a weather request."""
    return "Course Weather Server. Data source: WeatherAPI. A key is needed for live weather."


@mcp.prompt()
def weather_briefing(city: str) -> str:
    """Prepare a reusable request for a short weather briefing."""
    return f"Use current weather for {city} to write a short briefing. Include the city, temperature and conditions."


**Run the server in terminal**

On the TAFE Windows computers, run [setup_mcp_ss.bat](practice/session5/setup_mcp_ss.bat) to prepare the Python environment. [Download weather_mcp_server.py](practice/session5/weather_mcp_server.py) into your working directory, enter your WeatherAPI key in the file, and activate the environment. Inspector requires Node.js 22.19.0 or newer, which includes `npx`.

**1. Inspector — stdio transport**

Run this command in your working directory. Inspector starts the server and connects through stdio:

```bat
npx @modelcontextprotocol/inspector python weather_mcp_server.py -- --transport stdio
```

Open the browser URL printed by Inspector, connect to the server, and call `get_weather` with `city` set to `Perth`.

**2. Inspector — http transport**

Start the server in Terminal 1:

```bat
python weather_mcp_server.py
```

Keep it running. The server uses Streamable HTTP at `http://127.0.0.1:8000/mcp`. In Terminal 2, run Inspector:

```bat
npx @modelcontextprotocol/inspector --server-url http://127.0.0.1:8000/mcp --transport http
```

Open the browser URL printed by Inspector, connect to the server, and test `get_weather`. In section 5, use the same HTTP server with the Python client or chatbot. Press **Ctrl+C** in each terminal to stop its process.

[FastMCP HTTP server guide](https://gofastmcp.com/deployment/running-server) · [MCP Inspector guide](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector)


## 5. Connect MCP to the chatbot

The chatbot sends messages to an LLM API and calls `get_weather` through MCP. The FastMCP server requests WeatherAPI data and returns it to the chatbot for the LLM's answer. The diagram shows the HTTP connection.

![The chatbot connects to an LLM API and a FastMCP weather server, which calls the Weather API.](assets/session5/mcp-weather-chatbot-connection.png)

### 5.1 MCP client

Set `MCP_TRANSPORT` to `"http"` or `"stdio"`. HTTP requires a running weather server; stdio starts the server automatically.


In [ ]:
import json
import sys
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

SERVER_URL = "http://127.0.0.1:8000/mcp"
MCP_TRANSPORT = "http"  # Choose "http" or "stdio" for the notebook examples.
SERVER_SCRIPT = "practice/session5/weather_mcp_server.py"
server_target = SERVER_URL if MCP_TRANSPORT == "http" else StdioTransport(
    command=sys.executable,
    args=[SERVER_SCRIPT, "--transport", "stdio"],
    keep_alive=False,  # Close the server subprocess after each client session.
)

def tool_schema(tool):
    """Read the input schema exposed by the installed MCP SDK."""
    if hasattr(tool, "input_schema"):
        return tool.input_schema
    return tool.inputSchema

async with Client(server_target) as client:
    weather_tools = await client.list_tools()  # Discover this server's tools.
    for tool in weather_tools:
        print(tool.name, json.dumps(tool_schema(tool), indent=2))


The client discovers `get_weather` and its input schema. Set the WeatherAPI key in the server file, then call the tool below. Restart an HTTP server after changing its file.


In [ ]:
def result_text(result):
    """Join the text content returned by the weather MCP tool."""
    return "\n".join(block.text for block in result.content if getattr(block, "type", None) == "text")

async with Client(server_target) as client:
    result = await client.call_tool("get_weather", {"city": "Perth"})  # Request current weather.
    print(result_text(result))


### 5.2 Integrate into chatbot

`MCPChatbot` groups the MCP connection, LLM settings, tools and conversation history. It reuses the streaming chat from [Sessions 3–4](<Session 3-4 Building a Chatbot with an LLM API.ipynb>).

Enter your API key and a tool-capable model. Change the OpenRouter URL if using another compatible provider.


In [ ]:
import asyncio
import requests

CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
CHAT_API_KEY = "YOUR_LLM_API_KEY"  # Enter your LLM provider's API key.
CHAT_MODEL = "YOUR_MODEL_ID"  # Select a model that supports tool calls.


Read streamed text and tool-call fragments from the LLM response.


In [ ]:
def read_stream_deltas(response):
    """Read text and tool fragments from the API's SSE data lines."""
    # A small chunk_size avoids buffering short replies before displaying them.
    for line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if not line.startswith("data:"):
            continue  # Skip blank lines and keep-alive comments.
        data = line[5:].strip()
        if data == "[DONE]":
            return
        chunk = json.loads(data)
        if chunk.get("error"):
            raise RuntimeError(chunk["error"])
        for choice in chunk.get("choices", []):
            if choice.get("finish_reason") not in (None, "stop", "tool_calls"):
                raise RuntimeError(f"Incomplete model response: {choice['finish_reason']}")
            yield choice.get("delta", {})
    raise RuntimeError("The model response stream ended before [DONE].")


**Three methods that connect the chatbot to MCP**

- `connect_and_run()` checks the settings and opens the FastMCP client with `async with self.client`. It discovers the tools once, stores their definitions in `self.available_tools` and runs `chat_loop()` inside the client context. The context closes when you type `quit`.
- `mcp_tools_to_llm()` calls `self.client.list_tools()`, which sends a `tools/list` request to the MCP server. It converts each tool's name, description and input schema into the LLM API's function format. `send_messages()` includes these definitions in the `tools` field of its requests.
- `execute_tool(tool_name, tool_args)` checks that the requested name is in `tool_names`. It calls `self.client.call_tool()`, which sends the name and arguments in a `tools/call` request. The method extracts the text from the MCP result and returns it to `chat()`.

**Follow a weather question through the client**

The chatbot is the host. It uses its FastMCP client to exchange MCP messages with the weather server over stdio or Streamable HTTP. It talks to the LLM API separately, using HTTP requests with streamed responses. Read the diagram from top to bottom.

![Sequence diagram showing the chatbot discovering tools from the MCP server, sending tool definitions to the LLM, calling the requested MCP tool and returning its result to the LLM for a final answer.](assets/session5/mcp-client-message-flow.png)

1. You ask **What is the weather in Perth?** The chatbot sends the conversation and available tool definitions to the LLM API. The LLM can respond with a `tool_calls` entry requesting `get_weather` with `{"city": "Perth"}`.
2. `chat()` reads the tool name and decodes the JSON arguments, then passes them to `execute_tool()`. The FastMCP client sends the call to the weather MCP server. The server runs `get_weather` and sends the result back to the client.
3. `chat()` adds a message with `role: tool` and the matching `tool_call_id`. It sends the updated conversation to the LLM, which can use the weather result to stream an answer. If the LLM asks for another tool call, `chat()` repeats the same exchange.


In [ ]:
class MCPChatbot:
    """Own the MCP connection, LLM settings and conversation history."""

    def __init__(self, transport="http", *, server_url=SERVER_URL,
                 server_script="weather_mcp_server.py", api_url=CHAT_URL,
                 api_key=CHAT_API_KEY, model=CHAT_MODEL,
                 system_prompt=None):
        self.transport = transport
        self.api_url, self.api_key, self.model = api_url, api_key, model
        server = server_url if transport == "http" else StdioTransport(
            command=sys.executable,  # Use this Python environment for the subprocess.
            args=[server_script, "--transport", "stdio"],
            keep_alive=False,  # Close the subprocess when the client disconnects.
        )
        self.client = Client(server)
        self.available_tools = []
        self.messages = [{"role": "system", "content": system_prompt or (
            "You are a weather assistant. Use Australian English. "
            "Do not invent weather data. Ask for a city when it is missing."
        )}]

    def validate_settings(self):
        """Check the transport, API endpoint, API key and model before connecting."""
        if self.transport not in ("http", "stdio"):
            raise ValueError("Choose http or stdio for the MCP transport.")
        for name, value in (
            ("CHAT_URL", self.api_url),
            ("CHAT_API_KEY", self.api_key),
            ("CHAT_MODEL", self.model),
        ):
            if not isinstance(value, str) or not value.strip() or value.strip().startswith("YOUR_"):
                raise ValueError(f"Replace {name} before starting the chatbot.")
        if not self.api_url.startswith(("https://", "http://")):
            raise ValueError("CHAT_URL must be an HTTP or HTTPS API endpoint.")

    async def mcp_tools_to_llm(self):
        """Convert discovered MCP tools to the LLM's function declaration format."""
        tools = await self.client.list_tools()
        return [
            {"type": "function", "function": {
                "name": tool.name,
                "description": tool.description or tool.name,
                "parameters": tool_schema(tool),
            }}
            for tool in tools
        ]

    async def execute_tool(self, tool_name, tool_args):
        """Check the discovered tool name, invoke MCP and return text to the LLM."""
        tool_names = {tool["function"]["name"] for tool in self.available_tools}
        if tool_name not in tool_names:
            raise ValueError(f"Tool was not discovered on this server: {tool_name}")
        result = await self.client.call_tool(tool_name, tool_args)
        return result_text(result)

    async def connect_and_run(self):
        """Discover tools once and keep the MCP connection open for the chat."""
        self.validate_settings()
        async with self.client:
            self.available_tools = await self.mcp_tools_to_llm()
            tool_names = {tool["function"]["name"] for tool in self.available_tools}
            if not tool_names:
                raise ValueError("The server did not publish any tools.")
            print("Connected to server with tools:", sorted(tool_names))
            await self.chat_loop()

    def send_messages(self, messages):
        """Print streamed text and return the assembled assistant message."""
        # stream requests incremental responses from the model API.
        payload = {"model": self.model, "messages": messages, "tools": self.available_tools, "stream": True}
        message = {"role": "assistant", "content": ""}
        tool_calls = {}
        with requests.post(
            self.api_url,
            headers={"Authorization": f"Bearer {self.api_key}"},
            json=payload,
            stream=True,  # Read the HTTP response incrementally.
            timeout=(10, 60),  # Connection timeout and wait between received data.
        ) as response:
            response.raise_for_status()
            response.encoding = "utf-8"
            for delta in read_stream_deltas(response):
                text = delta.get("content")
                if text:
                    if not message["content"]:
                        print("Assistant: ", end="", flush=True)
                    print(text, end="", flush=True)
                    message["content"] += text

                # Group fragments by index; parallel tool calls may be interleaved.
                for fragment in delta.get("tool_calls") or []:
                    index = fragment.get("index", 0)
                    if index not in tool_calls:
                        tool_calls[index] = {
                            "id": "",
                            "type": "function",
                            "function": {"name": "", "arguments": ""},
                        }
                    call = tool_calls[index]
                    for key, value in fragment.items():
                        if key not in ("index", "function"):
                            # Preserve provider metadata, including thought signatures.
                            call[key] = value
                    function = fragment.get("function") or {}
                    for key in ("name", "arguments"):
                        call["function"][key] += function.get(key) or ""
        if message["content"]:
            print(flush=True)
        if tool_calls:
            message["tool_calls"] = [tool_calls[index] for index in sorted(tool_calls)]
        return message

    async def chat(self, user_message: str) -> str:
        """Complete a chat turn, running requested tools through MCP."""
        pending = self.messages + [{"role": "user", "content": user_message}]
        while True:
            # Keep the MCP event loop responsive while requests reads the LLM stream.
            response = await asyncio.to_thread(self.send_messages, pending)
            pending.append(response)  # Preserve the text, tool calls and their IDs.
            if not response.get("tool_calls"):
                break  # A text answer with no tool calls completes the turn.
            for call in response["tool_calls"]:
                tool_name = call["function"]["name"]
                tool_args = json.loads(call["function"]["arguments"])  # Read the JSON arguments.
                print(f"Calling tool {tool_name} with args {tool_args}", flush=True)
                result = await self.execute_tool(tool_name, tool_args)  # Invoke the MCP tool.
                pending.append({
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "content": result,  # Return the MCP text to the LLM.
                })

        self.messages = pending  # Save the complete turn after the requests succeed.
        return response["content"]

    async def chat_loop(self):
        """Read terminal input and complete turns until the user types quit."""
        print("\nMCP Chatbot Started!\n")
        print("Type your queries or 'quit' to exit.")
        while True:
            user_message = (await asyncio.to_thread(input, "You: ")).strip()
            if user_message.lower() == "quit":
                print("Chat ended.")
                return
            if user_message:
                await self.chat(user_message)


Create the chatbot. On startup, it prints the available tools and the prompt to enter a query or `quit`.


In [ ]:
chatbot = MCPChatbot(
    transport=MCP_TRANSPORT,  # Reuse the transport selected in section 5.1.
    server_script=SERVER_SCRIPT,
)


### 5.3 Run in terminal

Save [mcp_chatbot.py](practice/session5/mcp_chatbot.py) and [weather_mcp_server.py](practice/session5/weather_mcp_server.py) in your working directory. Enter the LLM settings and WeatherAPI key in their respective files. Activate the Python environment.

**HTTP — two terminals**

Terminal 1:

```bat
python weather_mcp_server.py --transport http
```

Terminal 2 connects to `http://127.0.0.1:8000/mcp`:

```bat
python mcp_chatbot.py --transport http
```

**stdio — one terminal**

The chatbot manages the server subprocess:

```bat
python mcp_chatbot.py --transport stdio
```

Ask about Perth, then Sydney. Type `quit` to exit; stop an HTTP server with **Ctrl+C**.

For notebook chat, select `MCP_TRANSPORT`, enter the LLM settings and enable the cell below.

For direct tool calls, use [mcp_client.py](practice/session5/mcp_client.py) (HTTP) or [mcp_client_stdio.py](practice/session5/mcp_client_stdio.py) (stdio).


In [ ]:
RUN_CHAT = False  # Set True to enable interactive LLM and weather API requests.

if RUN_CHAT:
    await chatbot.connect_and_run()  # Use await in a notebook that already has an event loop.
else:
    print("Interactive chat is off. Set RUN_CHAT to True when you are ready.")


## 6. Multi-tool MCP chatbot

This chatbot combines weather enquiries and research-paper search in one conversation. The `MultiToolChatbot` class inherits the streaming chat and tool handling from `MCPChatbot`, using four tools over HTTP or stdio.

| Tool | Functionality |
|---|---|
| `get_weather(city)` | Current temperature and conditions |
| `get_forecast(city, days=1)` | Daily minimum and maximum temperatures for one to three days |
| `get_history(city, dt)` | Daily recorded temperatures for a date in `YYYY-MM-DD` format |
| `search_papers(query, max_results=5)` | arXiv titles, summaries and source links; one to ten results |

**Download and configure**

- [mcp_chatbot.py](practice/session5/mcp_chatbot.py): provides the shared `MCPChatbot` class.
- [multi_tool_mcp_server.py](practice/session5/multi_tool_mcp_server.py): enter your WeatherAPI key in `WEATHER_API_KEY`.
- [mcp_multi_tool_chatbot.py](practice/session5/mcp_multi_tool_chatbot.py): enter the LLM `CHAT_API_KEY` and a tool-capable `CHAT_MODEL`, for your chosen LLM provider.

Save all three files in the same working directory. The existing `setup_mcp_ss.bat` installs `fastmcp`, `requests` and `arxiv`. No arXiv API key is needed. Forecast and historical access depend on your WeatherAPI plan; choose a supported past date for history requests.

**stdio — one terminal**

Activate the Python environment and run:

```bat
python mcp_multi_tool_chatbot.py --transport stdio
```

The chatbot starts and closes the multi-tool server subprocess.

**HTTP — two terminals**

Start the multi-tool server in Terminal 1:

```bat
python multi_tool_mcp_server.py --transport http
```

In Terminal 2, connect the chatbot to `http://127.0.0.1:8001/mcp`:

```bat
python mcp_multi_tool_chatbot.py --transport http
```

Type `quit` to end the chat. For HTTP, press **Ctrl+C** in Terminal 1 to stop the server.

**Try these requests**

1. “What is the current weather in Perth?” — `get_weather`.
2. “Give me a three-day forecast for Sydney.” — `get_forecast`.
3. “What was the weather in Perth yesterday?” — `get_history`.
4. “Find two arXiv papers about weather forecasting.” — `search_papers`.
5. “Compare current weather in Perth and Sydney.” — two `get_weather` calls.

Watch the printed tool names and arguments. Follow up with another city, date or topic to see how the chatbot keeps the conversation history and chooses a different tool.
